# CWL

[`cwltool`](https://github.com/common-workflow-language/cwltool) is the reference implementation of the [Common Workflow Language (CWL)](https://www.commonwl.org/), a standard for describing command-line based workflows. Unlike the other engines in this repository, CWL has no Python API: workflows are described in YAML/JSON documents and executed by the external `cwltool` command. This notebook converts the arithmetic `workflow.json` into an equivalent set of CWL files and executes them with `cwltool`.

In [1]:
import pickle

In [2]:
from python_workflow_definition.cwl import write_workflow

`write_workflow` reads the existing `workflow.json` and generates the corresponding CWL description: one `CommandLineTool` `.cwl` file per function node, a `workflow.cwl` file that wires the steps together, and a `workflow.yml` file listing the input values (each serialized to its own pickle file). Every generated tool invokes `python -m python_workflow_definition.cwl`, which loads the target function from `workflow.py` and calls it with the pickled inputs.

In [3]:
write_workflow(file_name="workflow.json")

Running `cwltool workflow.cwl workflow.yml` executes the steps in dependency order, passing intermediate results between steps as pickle files, and prints the location of the final output file.

In [4]:
! cwltool workflow.cwl workflow.yml

/srv/conda/envs/notebook/bin/cwltool:11: DeprecationWarning: Nesting argument groups is deprecated.
  sys.exit(run())
INFO /srv/conda/envs/notebook/bin/cwltool 3.1.20250110105449
INFO Resolved 'workflow.cwl' to 'file:///home/jovyan/example_workflows/arithmetic/workflow.cwl'
INFO [workflow ] start
INFO [workflow ] starting step get_prod_and_div_0
INFO [step get_prod_and_div_0] start
INFO [job get_prod_and_div_0] /tmp/_1apt559$ python \
    -m \
    python_workflow_definition.cwl \
    --workflowfile=/tmp/dyuxg6ka/stgf337bac5-2f8e-4489-bd51-83491b1d5f0e/workflow.py \
    --function=workflow.get_prod_and_div \
    --arg_x=/tmp/dyuxg6ka/stgd87c2fe4-ca8e-4031-9af0-751d6944cbe7/x.pickle \
    --arg_y=/tmp/dyuxg6ka/stg7bcb8ced-0e5b-442e-a0e0-2b3117afe2d8/y.pickle
INFO [job get_prod_and_div_0] completed success
INFO [step get_prod_and_div_0] completed success
INFO [workflow ] starting step get_sum_1
INFO [step get_sum_1] start
INFO [job get_sum_1] /tmp/ysau_yra$ python \
    -m \
    python_wo

The workflow's final output is written to `result.pickle`; unpickling it gives the same numeric result produced by the other engines.

In [5]:
with open("result.pickle", "rb") as f:
    print(pickle.load(f))

6.25
